# ArmsFlow: India's supplier patterns

Historical imports of major conventional arms, 2010–2025. Source: [SIPRI](https://armstransfers.sipri.org/), downloaded 6 November 2026. TIV is transfer volume, not money or weapon counts. Run the project analysis command before this notebook.

In [ ]:
from pathlib import Path
import sys
import duckdb
from IPython.display import Image, display
ROOT = Path.cwd()
if not (ROOT / 'data/processed/armsflow.duckdb').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DB = ROOT / 'data/processed/armsflow.duckdb'
assert DB.is_file()
print('Loaded the validated India database.')

## Five-year supplier shares
Aggregate delivered TIV first; divide each supplier's period total by the recipient's total. Unknown suppliers remain in the denominator.

In [ ]:
with duckdb.connect(str(DB), read_only=True) as con:
    rows = con.execute("""SELECT period_id, e.display_label, round(share_percent, 2) AS share_percent
        FROM metric_period_shares s JOIN entity e ON e.entity_id=s.supplier_id
        WHERE period_kind='five_year' AND supplier_id IN ('russia','france','israel','united_states')
        ORDER BY start_year, share_percent DESC""").fetchall()
for period, supplier, share in rows:
    print(f'{period}  {supplier:15}  {share:6.2f}%')

In [ ]:
with duckdb.connect(str(DB), read_only=True) as con:
    rows = con.execute("SELECT period_id, total_tiv_million, top_three_share, hhi FROM metric_period_summary WHERE period_kind='five_year' ORDER BY start_year").fetchall()
for period, total, top3, hhi in rows:
    print(f'{period}: {total:,.2f} million TIV | top 3 {100*top3:.1f}% | HHI {hhi:.3f}')

## Changes and validation
Percentage-point changes compare shares; volume changes compare TIV totals. Supplier-mix distance is half the sum of absolute share differences.

In [ ]:
from pipeline.build_metrics import verify
with duckdb.connect(str(DB), read_only=True) as con:
    checks = verify(con)
    changes = con.execute("SELECT comparison_id, supplier_mix_distance, volume_change_percent FROM metric_period_changes WHERE comparison_kind <> 'annual' ORDER BY comparison_id").fetchall()
print(f'{len(checks)} metric checks passed.')
for comparison, distance, volume in changes:
    print(f'{comparison}: mix distance {distance:.3f}; volume change {volume:+.2f}%')

In [ ]:
display(Image(filename=str(ROOT / 'reports/india_supplier_analysis.png')))
display(Image(filename=str(ROOT / 'reports/india_annual_supplier_shares.png')))

## Interpretation limits

This is a retrospective India pilot using one revised SIPRI snapshot. Source estimates and unresolved field conflicts remain. It is not a current inventory, a causal explanation of geopolitical events or an ML forecast. Incomplete coverage invalidates share metrics; a complete zero-total period has undefined shares. Rebuild with `python -m pipeline.analyze --notebook`.